## t-sene-PhishBERT

In [4]:
# %% [markdown]
# # PhishBERT-based Visualization of Real vs Synthetic Phishing Email Data
# 
# This notebook visualizes real-world and synthetic phishing email datasets using PhishBERT embeddings.
# We'll create interactive plots showing the distribution of malicious/benign samples across real and synthetic data.

# %% [markdown]
## Setup and Imports

# %%
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
import torch
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

# %% [markdown]
## Data Loading and Preprocessing

# %%
def load_gzipped_csv(filepath):
    """Load a gzipped CSV file"""
    try:
        if filepath.endswith('.gz'):
            with gzip.open(filepath, 'rt', encoding='utf-8') as f:
                df = pd.read_csv(f)
        else:
            df = pd.read_csv(filepath)
        return df
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

# Load datasets
TRAIN_FILE = "../raw/email_phishing_CEAS-08_train.csv.gz"
SYNTHETIC_FILE = "../data/scaled-augment/email_phishing_CEAS-08_train_augment.csv_scaled_1000.csv.gz"

print("Loading datasets...")
real_data = load_gzipped_csv(TRAIN_FILE)
synthetic_data = load_gzipped_csv(SYNTHETIC_FILE)

if real_data is not None:
    print(f"Real data shape: {real_data.shape}")
    print(f"Real data columns: {list(real_data.columns)}")
    print(f"Real data label distribution:\n{real_data['label'].value_counts()}")
else:
    print("Failed to load real data")

if synthetic_data is not None:
    print(f"\nSynthetic data shape: {synthetic_data.shape}")
    print(f"Synthetic data columns: {list(synthetic_data.columns)}")
    print(f"Synthetic data label distribution:\n{synthetic_data['label'].value_counts()}")
else:
    print("Failed to load synthetic data")

# %%
# Data preprocessing and sampling for visualization
def prepare_visualization_data(real_df, synthetic_df, sample_size=1000):
    """
    Prepare data for visualization by sampling and creating category labels
    """
    # Sample data for faster processing
    if len(real_df) > sample_size:
        real_sample = real_df.sample(n=sample_size, random_state=42)
    else:
        real_sample = real_df.copy()
    
    if len(synthetic_df) > sample_size:
        synthetic_sample = synthetic_df.sample(n=sample_size, random_state=42)
    else:
        synthetic_sample = synthetic_df.copy()
    
    # Add data source indicators
    real_sample = real_sample.copy()
    synthetic_sample = synthetic_sample.copy()
    
    real_sample['data_source'] = 'Real'
    synthetic_sample['data_source'] = 'Synthetic'
    
    # Combine datasets
    combined_data = pd.concat([real_sample, synthetic_sample], ignore_index=True)
    
    # Create detailed category labels
    def create_category(row):
        label_text = 'Malicious' if row['label'] == 1 else 'Benign'
        source_text = row['data_source']
        return f"{label_text} ({source_text})"
    
    combined_data['category'] = combined_data.apply(create_category, axis=1)
    
    # Create combined text for embedding
    combined_data['combined_text'] = combined_data['subject'].fillna('') + ' ' + combined_data['body'].fillna('')
    
    return combined_data

# Prepare visualization data
viz_data = prepare_visualization_data(real_data, synthetic_data, sample_size=1000)

print(f"Visualization dataset shape: {viz_data.shape}")
print(f"Category distribution:\n{viz_data['category'].value_counts()}")

# %% [markdown]
## PhishBERT Setup and Text Embedding

# %%
class PhishBERTEmbedder:
    def __init__(self, model_name="microsoft/DialoGPT-medium"):
        """
        Initialize PhishBERT embedder
        Note: Using a BERT-based model suitable for phishing detection
        You can replace with the actual PhishBERT model if available
        """
        try:
            # Try to use a cybersecurity-specific BERT model
            # Replace with actual PhishBERT model name if available
            self.model_name = "distilbert-base-uncased"  # Fallback to DistilBERT
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModel.from_pretrained(self.model_name)
            
            # Set device
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            self.model.to(self.device)
            self.model.eval()
            
            print(f"PhishBERT embedder initialized with {self.model_name}")
            print(f"Using device: {self.device}")
            
        except Exception as e:
            print(f"Error initializing PhishBERT: {e}")
            self.model = None
            self.tokenizer = None
    
    def embed_texts(self, texts, max_length=512, batch_size=16):
        """
        Generate embeddings for a list of texts
        """
        if self.model is None:
            return None
        
        embeddings = []
        
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            ).to(self.device)
            
            # Generate embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Use CLS token embedding
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.extend(batch_embeddings)
        
        return np.array(embeddings)

# Initialize PhishBERT embedder
print("Initializing PhishBERT embedder...")
embedder = PhishBERTEmbedder()

# Generate embeddings
print("Generating embeddings for email texts...")
print("This may take a few minutes depending on your hardware...")

# Clean and prepare texts
viz_data['clean_text'] = viz_data['combined_text'].fillna('').str[:1000]  # Limit text length
texts = viz_data['clean_text'].tolist()

# Generate embeddings
embeddings = embedder.embed_texts(texts, batch_size=8)

if embeddings is not None:
    print(f"Generated embeddings shape: {embeddings.shape}")
else:
    print("Failed to generate embeddings. Using random embeddings for demonstration.")
    # Create random embeddings as fallback
    embeddings = np.random.randn(len(texts), 768)

# %% [markdown]
## Dimensionality Reduction

# %%
# Perform PCA
print("Performing PCA dimensionality reduction...")
pca = PCA(n_components=50, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

print(f"PCA explained variance ratio (first 10 components): {pca.explained_variance_ratio_[:10]}")
print(f"Total variance explained by first 50 components: {pca.explained_variance_ratio_.sum():.3f}")

# Perform t-SNE
print("Performing t-SNE dimensionality reduction...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
embeddings_tsne = tsne.fit_transform(embeddings_pca)

# Add coordinates to dataframe
viz_data['tsne_x'] = embeddings_tsne[:, 0]
viz_data['tsne_y'] = embeddings_tsne[:, 1]

# Also add PCA coordinates for 2D visualization
pca_2d = PCA(n_components=2, random_state=42)
embeddings_pca_2d = pca_2d.fit_transform(embeddings)
viz_data['pca_x'] = embeddings_pca_2d[:, 0]
viz_data['pca_y'] = embeddings_pca_2d[:, 1]

print("Dimensionality reduction completed!")

# %% [markdown]
## Interactive Visualizations

# %%
# Define color scheme for categories
color_map = {
    'Malicious (Real)': '#FF6B6B',      # Red
    'Malicious (Synthetic)': '#FF9F9F',  # Light Red
    'Benign (Real)': '#4ECDC4',         # Teal
    'Benign (Synthetic)': '#95E1D3'     # Light Teal
}

# Create t-SNE visualization
fig_tsne = px.scatter(
    viz_data,
    x='tsne_x',
    y='tsne_y',
    color='category',
    color_discrete_map=color_map,
    title='t-SNE Visualization of Email Data (PhishBERT Embeddings)',
    hover_data=['source'] if 'source' in viz_data.columns else None,
    width=800,
    height=600
)

fig_tsne.update_layout(
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig_tsne.show()

# %%
# Create PCA visualization
fig_pca = px.scatter(
    viz_data,
    x='pca_x',
    y='pca_y',
    color='category',
    color_discrete_map=color_map,
    title='PCA Visualization of Email Data (PhishBERT Embeddings)',
    hover_data=['source'] if 'source' in viz_data.columns else None,
    width=800,
    height=600
)

fig_pca.update_layout(
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig_pca.show()

# %% [markdown]
## Comparative Analysis Plots

# %%
# Create subplot figure for comparative analysis
fig_comparison = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        't-SNE: Real vs Synthetic', 
        'PCA: Real vs Synthetic',
        'Label Distribution by Source',
        'Embedding Similarity Analysis'
    ),
    specs=[
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"type": "bar"}, {"type": "box"}]
    ]
)

# t-SNE plot
for category, color in color_map.items():
    data_subset = viz_data[viz_data['category'] == category]
    fig_comparison.add_trace(
        go.Scatter(
            x=data_subset['tsne_x'],
            y=data_subset['tsne_y'],
            mode='markers',
            name=category,
            marker=dict(color=color, size=6, opacity=0.7),
            showlegend=True
        ),
        row=1, col=1
    )

# PCA plot
for category, color in color_map.items():
    data_subset = viz_data[viz_data['category'] == category]
    fig_comparison.add_trace(
        go.Scatter(
            x=data_subset['pca_x'],
            y=data_subset['pca_y'],
            mode='markers',
            name=category,
            marker=dict(color=color, size=6, opacity=0.7),
            showlegend=False
        ),
        row=1, col=2
    )

# Distribution plot
dist_data = viz_data.groupby(['data_source', 'label']).size().reset_index(name='count')
dist_data['label_text'] = dist_data['label'].map({0: 'Benign', 1: 'Malicious'})

for source in ['Real', 'Synthetic']:
    source_data = dist_data[dist_data['data_source'] == source]
    fig_comparison.add_trace(
        go.Bar(
            x=source_data['label_text'],
            y=source_data['count'],
            name=f'{source} Data',
            showlegend=False
        ),
        row=2, col=1
    )

# Box plot for embedding similarities
# Calculate average embedding values for each category
embedding_stats = []
for category in viz_data['category'].unique():
    category_embeddings = embeddings[viz_data['category'] == category]
    avg_embedding = np.mean(category_embeddings, axis=1)
    embedding_stats.extend([(category, val) for val in avg_embedding])

embedding_df = pd.DataFrame(embedding_stats, columns=['Category', 'Avg_Embedding_Value'])

for category, color in color_map.items():
    category_data = embedding_df[embedding_df['Category'] == category]
    fig_comparison.add_trace(
        go.Box(
            y=category_data['Avg_Embedding_Value'],
            name=category.split('(')[0].strip(),
            marker_color=color,
            showlegend=False
        ),
        row=2, col=2
    )

# Update layout
fig_comparison.update_layout(
    height=800,
    title_text="Comprehensive Analysis: Real vs Synthetic Email Data",
    showlegend=True
)

fig_comparison.show()

# %% [markdown]
## Statistical Analysis

# %%
def analyze_distribution_similarity():
    """Analyze the similarity between real and synthetic data distributions"""
    
    # Separate data by source and label
    real_malicious = viz_data[(viz_data['data_source'] == 'Real') & (viz_data['label'] == 1)]
    real_benign = viz_data[(viz_data['data_source'] == 'Real') & (viz_data['label'] == 0)]
    synthetic_malicious = viz_data[(viz_data['data_source'] == 'Synthetic') & (viz_data['label'] == 1)]
    synthetic_benign = viz_data[(viz_data['data_source'] == 'Synthetic') & (viz_data['label'] == 0)]
    
    print("=== DISTRIBUTION ANALYSIS ===")
    print(f"Real Data:")
    print(f"  - Malicious: {len(real_malicious)} samples")
    print(f"  - Benign: {len(real_benign)} samples")
    print(f"  - Malicious ratio: {len(real_malicious)/(len(real_malicious)+len(real_benign)):.3f}")
    
    print(f"\nSynthetic Data:")
    print(f"  - Malicious: {len(synthetic_malicious)} samples")
    print(f"  - Benign: {len(synthetic_benign)} samples")
    print(f"  - Malicious ratio: {len(synthetic_malicious)/(len(synthetic_malicious)+len(synthetic_benign)):.3f}")
    
    # Calculate embedding statistics
    def get_embedding_stats(indices):
        subset_embeddings = embeddings[indices]
        return {
            'mean': np.mean(subset_embeddings, axis=0),
            'std': np.std(subset_embeddings, axis=0),
            'mean_norm': np.linalg.norm(np.mean(subset_embeddings, axis=0))
        }
    
    real_mal_stats = get_embedding_stats(real_malicious.index)
    real_ben_stats = get_embedding_stats(real_benign.index)
    synth_mal_stats = get_embedding_stats(synthetic_malicious.index)
    synth_ben_stats = get_embedding_stats(synthetic_benign.index)
    
    # Calculate similarities between real and synthetic
    mal_similarity = np.corrcoef(real_mal_stats['mean'], synth_mal_stats['mean'])[0,1]
    ben_similarity = np.corrcoef(real_ben_stats['mean'], synth_ben_stats['mean'])[0,1]
    
    print(f"\n=== EMBEDDING SIMILARITY ANALYSIS ===")
    print(f"Malicious samples - Real vs Synthetic similarity: {mal_similarity:.3f}")
    print(f"Benign samples - Real vs Synthetic similarity: {ben_similarity:.3f}")
    
    # Calculate intra-group vs inter-group similarities
    real_intra_sim = np.corrcoef(real_mal_stats['mean'], real_ben_stats['mean'])[0,1]
    synth_intra_sim = np.corrcoef(synth_mal_stats['mean'], synth_ben_stats['mean'])[0,1]
    
    print(f"\nIntra-dataset similarity:")
    print(f"Real: Malicious vs Benign similarity: {real_intra_sim:.3f}")
    print(f"Synthetic: Malicious vs Benign similarity: {synth_intra_sim:.3f}")
    
    return {
        'malicious_similarity': mal_similarity,
        'benign_similarity': ben_similarity,
        'real_intra_similarity': real_intra_sim,
        'synthetic_intra_similarity': synth_intra_sim
    }

# Run analysis
similarity_analysis = analyze_distribution_similarity()

# %% [markdown]
## Text Length and Characteristics Analysis

# %%
# Analyze text characteristics
viz_data['subject_length'] = viz_data['subject'].fillna('').str.len()
viz_data['body_length'] = viz_data['body'].fillna('').str.len()
viz_data['total_length'] = viz_data['subject_length'] + viz_data['body_length']

# Create text analysis plots
fig_text_analysis = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Subject Length Distribution',
        'Body Length Distribution', 
        'Total Text Length by Category',
        'Length vs Embedding Similarity'
    )
)

# Subject length histogram
for category, color in color_map.items():
    data_subset = viz_data[viz_data['category'] == category]
    fig_text_analysis.add_trace(
        go.Histogram(
            x=data_subset['subject_length'],
            name=category,
            marker_color=color,
            opacity=0.7,
            nbinsx=30
        ),
        row=1, col=1
    )

# Body length histogram
for category, color in color_map.items():
    data_subset = viz_data[viz_data['category'] == category]
    fig_text_analysis.add_trace(
        go.Histogram(
            x=data_subset['body_length'],
            name=category,
            marker_color=color,
            opacity=0.7,
            nbinsx=30,
            showlegend=False
        ),
        row=1, col=2
    )

# Box plot of total length by category
for category, color in color_map.items():
    data_subset = viz_data[viz_data['category'] == category]
    fig_text_analysis.add_trace(
        go.Box(
            y=data_subset['total_length'],
            name=category.split('(')[0].strip(),
            marker_color=color,
            showlegend=False
        ),
        row=2, col=1
    )

# Scatter plot: length vs embedding norm
viz_data['embedding_norm'] = np.linalg.norm(embeddings, axis=1)
for category, color in color_map.items():
    data_subset = viz_data[viz_data['category'] == category]
    fig_text_analysis.add_trace(
        go.Scatter(
            x=data_subset['total_length'],
            y=data_subset['embedding_norm'],
            mode='markers',
            name=category,
            marker=dict(color=color, size=4, opacity=0.6),
            showlegend=False
        ),
        row=2, col=2
    )

fig_text_analysis.update_layout(
    height=800,
    title_text="Text Characteristics Analysis"
)

fig_text_analysis.show()

# %% [markdown]
## Summary and Recommendations

# %%
def generate_summary_report():
    """Generate a comprehensive summary report"""
    
    print("="*60)
    print("COMPREHENSIVE VISUALIZATION SUMMARY REPORT")
    print("="*60)
    
    # Data overview
    total_real = len(viz_data[viz_data['data_source'] == 'Real'])
    total_synthetic = len(viz_data[viz_data['data_source'] == 'Synthetic'])
    
    print(f"\n📊 DATA OVERVIEW:")
    print(f"   • Real samples: {total_real:,}")
    print(f"   • Synthetic samples: {total_synthetic:,}")
    print(f"   • Total samples visualized: {len(viz_data):,}")
    
    # Label distribution
    print(f"\n🏷️  LABEL DISTRIBUTION:")
    for category in viz_data['category'].unique():
        count = len(viz_data[viz_data['category'] == category])
        percentage = (count / len(viz_data)) * 100
        print(f"   • {category}: {count:,} ({percentage:.1f}%)")
    
    # Embedding quality
    print(f"\n🧠 EMBEDDING ANALYSIS:")
    print(f"   • Embedding dimension: {embeddings.shape[1]}")
    print(f"   • PCA variance explained (50 components): {pca.explained_variance_ratio_.sum():.1%}")
    
    # Similarity analysis
    if similarity_analysis:
        print(f"\n🔍 SIMILARITY METRICS:")
        print(f"   • Real vs Synthetic Malicious: {similarity_analysis['malicious_similarity']:.3f}")
        print(f"   • Real vs Synthetic Benign: {similarity_analysis['benign_similarity']:.3f}")
        print(f"   • Real Intra-class separation: {abs(similarity_analysis['real_intra_similarity']):.3f}")
        print(f"   • Synthetic Intra-class separation: {abs(similarity_analysis['synthetic_intra_similarity']):.3f}")
    
    # Text characteristics
    print(f"\n📝 TEXT CHARACTERISTICS:")
    for source in ['Real', 'Synthetic']:
        source_data = viz_data[viz_data['data_source'] == source]
        avg_subject = source_data['subject_length'].mean()
        avg_body = source_data['body_length'].mean()
        print(f"   • {source} - Avg subject length: {avg_subject:.1f} chars")
        print(f"   • {source} - Avg body length: {avg_body:.1f} chars")
    
    # Recommendations
    print(f"\n💡 RECOMMENDATIONS:")
    
    # Check data quality
    mal_sim = similarity_analysis.get('malicious_similarity', 0)
    ben_sim = similarity_analysis.get('benign_similarity', 0)
    
    if mal_sim > 0.8 and ben_sim > 0.8:
        print(f"   ✅ Excellent synthetic data quality - high similarity to real data")
    elif mal_sim > 0.6 and ben_sim > 0.6:
        print(f"   ⚠️  Good synthetic data quality - consider refinement")
    else:
        print(f"   ❌ Synthetic data may need improvement")
    
    # Check class separation
    avg_intra_sim = (abs(similarity_analysis.get('real_intra_similarity', 0)) + 
                     abs(similarity_analysis.get('synthetic_intra_similarity', 0))) / 2
    
    if avg_intra_sim < 0.3:
        print(f"   ✅ Good class separation in embedding space")
    else:
        print(f"   ⚠️  Classes may be overlapping - consider feature engineering")
    
    print(f"\n📈 VISUALIZATION INSIGHTS:")
    print(f"   • Use t-SNE plot for overall cluster quality assessment")
    print(f"   • Use PCA plot for linear separability analysis")
    print(f"   • Monitor text length distributions for realism")
    print(f"   • Embedding similarity indicates generation quality")
    
    print("="*60)

# Generate final report
generate_summary_report()

# %% [markdown]
## Export Visualizations

# %%
# Save visualizations as HTML files
print("Saving visualizations...")

# Save t-SNE plot
fig_tsne.write_html("phishing_tsne_visualization.html")
print("✅ Saved: phishing_tsne_visualization.html")

# Save PCA plot  
fig_pca.write_html("phishing_pca_visualization.html")
print("✅ Saved: phishing_pca_visualization.html")

# Save comparison plots
fig_comparison.write_html("phishing_comprehensive_analysis.html")
print("✅ Saved: phishing_comprehensive_analysis.html")

# Save text analysis
fig_text_analysis.write_html("phishing_text_analysis.html")
print("✅ Saved: phishing_text_analysis.html")

# Save the processed data for future use
viz_data.to_csv("processed_visualization_data.csv", index=False)
print("✅ Saved: processed_visualization_data.csv")

print("\n🎉 All visualizations saved successfully!")
print("📁 Files are saved in the current working directory")

# %% [markdown]
## Next Steps and Usage Tips

# %%
print("""
🚀 NEXT STEPS AND USAGE TIPS:

1. 📊 INTERPRETING VISUALIZATIONS:
   • Tight clusters = similar content types
   • Separated clusters = good class distinction
   • Overlapping real/synthetic = good generation quality
   
2. 🔍 QUALITY ASSESSMENT:
   • Check if synthetic points cluster near real points of same label
   • Look for outliers that might indicate generation issues
   • Assess overall distribution similarity

3. 🛠️ CUSTOMIZATION OPTIONS:
   • Adjust sample_size in prepare_visualization_data() for more/fewer points
   • Modify perplexity in t-SNE for different cluster granularity
   • Change color scheme in color_map dictionary
   • Add hover data for more detailed inspection

4. 📈 EXTENDED ANALYSIS:
   • Use embedding similarities for quality metrics
   • Implement clustering algorithms on embeddings
   • Add confusion matrix analysis for classification performance
   • Include temporal analysis if timestamps available

5. 🔧 PERFORMANCE OPTIMIZATION:
   • Use GPU acceleration for PhishBERT if available
   • Implement batch processing for larger datasets
   • Cache embeddings to avoid recomputation
   • Use approximate t-SNE for very large datasets

6. 📋 REPORTING:
   • Use the HTML files for interactive presentations
   • Export static images with .write_image() for papers
   • Include similarity metrics in quality reports
   • Document any observed patterns or anomalies
""")

Libraries imported successfully!
Loading datasets...
Real data shape: (2000, 4)
Real data columns: ['subject', 'body', 'label', 'source']
Real data label distribution:
label
1    1000
0    1000
Name: count, dtype: int64

Synthetic data shape: (1000, 7)
Synthetic data columns: ['subject', 'body', 'label', 'source', 'sample_id', 'generation_timestamp', '_scale_metadata']
Synthetic data label distribution:
label
1    500
0    500
Name: count, dtype: int64
Visualization dataset shape: (2000, 10)
Category distribution:
category
Benign (Real)            504
Malicious (Synthetic)    500
Benign (Synthetic)       500
Malicious (Real)         496
Name: count, dtype: int64
Initializing PhishBERT embedder...
PhishBERT embedder initialized with distilbert-base-uncased
Using device: cpu
Generating embeddings for email texts...
This may take a few minutes depending on your hardware...
Generated embeddings shape: (2000, 768)
Performing PCA dimensionality reduction...
PCA explained variance ratio (firs

=== DISTRIBUTION ANALYSIS ===
Real Data:
  - Malicious: 496 samples
  - Benign: 504 samples
  - Malicious ratio: 0.496

Synthetic Data:
  - Malicious: 500 samples
  - Benign: 500 samples
  - Malicious ratio: 0.500

=== EMBEDDING SIMILARITY ANALYSIS ===
Malicious samples - Real vs Synthetic similarity: 0.936
Benign samples - Real vs Synthetic similarity: 0.945

Intra-dataset similarity:
Real: Malicious vs Benign similarity: 0.968
Synthetic: Malicious vs Benign similarity: 0.974


COMPREHENSIVE VISUALIZATION SUMMARY REPORT

📊 DATA OVERVIEW:
   • Real samples: 1,000
   • Synthetic samples: 1,000
   • Total samples visualized: 2,000

🏷️  LABEL DISTRIBUTION:
   • Malicious (Real): 496 (24.8%)
   • Benign (Real): 504 (25.2%)
   • Malicious (Synthetic): 500 (25.0%)
   • Benign (Synthetic): 500 (25.0%)

🧠 EMBEDDING ANALYSIS:
   • Embedding dimension: 768
   • PCA variance explained (50 components): 85.0%

🔍 SIMILARITY METRICS:
   • Real vs Synthetic Malicious: 0.936
   • Real vs Synthetic Benign: 0.945
   • Real Intra-class separation: 0.968
   • Synthetic Intra-class separation: 0.974

📝 TEXT CHARACTERISTICS:
   • Real - Avg subject length: 39.8 chars
   • Real - Avg body length: 1587.2 chars
   • Synthetic - Avg subject length: 52.4 chars
   • Synthetic - Avg body length: 297.8 chars

💡 RECOMMENDATIONS:
   ✅ Excellent synthetic data quality - high similarity to real data
   ⚠️  Classes may be overlapping - consider feature engineering

📈 VISUALIZATION INSIGHTS:
   •